In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np
from scipy.spatial.distance import cosine

Today we will focus on finding similarities between documents. For this purpose, we will compare the content of these documents. The same techniques can be used for a query in a search engine. Then simply we can treat the query like another document, calculate similarities and return the most similar documents.

In [2]:
documents = ['Machine Learning',
 'Five Advanced Plots in Python - Matplotlib',
 'How to Make your Computer Talk with Python',
 'Anomaly Detection on Servo Drives',
 'Key takeaways from Kaggle’s most recent time series competition - Ventilator Pressure Prediction',
 'Animated Mathematical Analysis',
 'How to Perform Speech Recognition with Python',
 'Beyond The Semesters: E04',
 'How to improve classification of e-commerce pages, incorporating multiple modalities',
 'Time Series Forecasting with ThymeBoost',
 'CHAPTER 2: Why I Chose Data Science!',
 'Training Provably-Robust Neural Networks',
 'Time Series Forecasting with ThymeBoost',
 'How to improve classification of e-commerce pages, incorporating multiple modalities',
 '5 Cute Features of CatBoost',
 'Variance Inflation Factor (VIF) and it’s relationship with multicollinearity&nbsp;.',
 'Beyond The Semesters: E04',
 'Efficient Digital Transformation - Particle Swarm Optimiser',
 'MEASURE OF ASYMMETRY',
 'What is linear regression? A quick cover with a tutorial',
 'Correlation VS Covariance: The easy way',
 'Are Recommender System harming us?',
 '1 Line of Python Code That Will Speed Up Your AI by Up to 6x',
 'If You Are Serious About Data Science Job. You Must Know These 3 Things.',
 'Recommender System With Machine Learning and Statistics',
 'Bias detection and mitigation in IBM AutoAI',
 'Data Engineering: Create your own Dataset',
 'Graph Neural Networks and Generalizable Models in Neuroscience',
 'Fastest Way of Deploying Your Machine Learning Models',
 'A Novel Approach to Integrate Speech Recognition into Authentication Systems',
 '3 Lessons Learned in Teaching Machine Learning for Earth Observation Techniques',
 'Vision Transformer in Galaxy Morphology Classification',
 'Exploring Methods of Deep Reinforcement Learning with NLP Applications',
 '6 Essential Tips to Solve Data Science Projects',
 'Data Science Interview Questions My Friends and I got asked recently (III)',
 'Understanding Uber’s Generative Teaching Networks',
 'How to achieve efficient large-batch training?',
 'How Parallelization and Large Batch Size Improve the Performance of Deep Neural Networks.',
 'Why You Need to Know the Inner Workings of Models',
 'Let’s Build A Simple Object Classification Task I']

In [3]:
CountVec = CountVectorizer(ngram_range=(1,1), stop_words='english')
CountData = CountVec.fit_transform(documents)

CountData

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 204 stored elements and shape (40, 150)>

The very basic way of storing information about documents is word count. Simply for each document we store an information how many times each word appears. It can be stored in an array, however, it's not the best option since it will be filled mostly with 0s. That's why it's stored in a sparse matrix, but we can expand it.

In [4]:
df=pd.DataFrame(CountData.toarray(), columns=CountVec.get_feature_names_out(), index=documents)
df

,6x,achieve,advanced,ai,analysis,animated,anomaly,applications,approach,asked,...,tutorial,uber,understanding,variance,ventilator,vif,vision,vs,way,workings
Machine Learning,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Five Advanced Plots in Python - Matplotlib,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
How to Make your Computer Talk with Python,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Anomaly Detection on Servo Drives,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Key takeaways from Kaggle’s most recent time series competition - Ventilator Pressure Prediction,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
Animated Mathematical Analysis,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
How to Perform Speech Recognition with Python,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Beyond The Semesters: E04,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"How to improve classification of e-commerce pages, incorporating multiple modalities",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Time Series Forecasting with ThymeBoost,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Task 1
We can reduce the size of an array, get rid of unnecesary words, and improve the quality of comparison by firstly preprocessing the docuemnts.
Check array size after stemming/lemmatization and without stop words

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import nltk, re

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mikol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mikol\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
# --- Preprocessing setup ---
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = re.sub(r'[^a-zA-Z]', ' ', text.lower())
    tokens = [word for word in text.split() if word not in stop_words]
    tokens = [lemmatizer.lemmatize(stemmer.stem(word)) for word in tokens]
    return " ".join(tokens)

# --- Apply preprocessing ---
cleaned_docs = [preprocess(doc) for doc in documents]

# --- Compare vocabulary sizes ---
cv_original = CountVectorizer(stop_words='english')
cv_cleaned = CountVectorizer()

original_mat = cv_original.fit_transform(documents)
cleaned_mat = cv_cleaned.fit_transform(cleaned_docs)

print(f"Original vocabulary size: {len(cv_original.get_feature_names_out())}")
print(f"After preprocessing vocabulary size: {len(cv_cleaned.get_feature_names_out())}")
print(f"Vocabulary reduced by: {len(cv_original.get_feature_names_out()) - len(cv_cleaned.get_feature_names_out())}")


Original vocabulary size: 150
After preprocessing vocabulary size: 147
Vocabulary reduced by: 3


## Task 2

Easy technique to compare two documents is a jaccard similarity.
$J={\frac {|A\cap B|}{|A\cup B|}}.$

Implement Jaccard similarity, and function finding closest document to a provided query. Test different queries

In [7]:
def jaccard(set1, set2):
    """Compute Jaccard similarity between two sets of words."""
    if not set1 or not set2:
        return 0.0
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    return len(intersection) / len(union)


def closest(query, df):
    """Find the document (row of df) most similar to query using Jaccard similarity."""
    # Tokenize query into lowercase words
    query_tokens = set(re.findall(r'\b[a-z]+\b', query.lower()))
    
    best_doc = None
    best_score = 0
    
    # Iterate over each document (row name = document text)
    for doc_name in df.index:
        doc_tokens = set(re.findall(r'\b[a-z]+\b', doc_name.lower()))
        score = jaccard(query_tokens, doc_tokens)
        
        if score > best_score:
            best_score = score
            best_doc = doc_name
    
    return f"Most similar: {best_doc}\nJaccard similarity: {best_score:.3f}"

<a href="https://ibb.co/k4rRpf9"><img src="https://i.ibb.co/GW1KXLt/ir4.jpg" alt="ir4" border="0"></a>

In [8]:
queries = [
    "python",
    "plot neural network",
    "plot neural networks",
    "ploting neural networks",
    "data science",
]
for q in queries:
    print(q)
    print(closest(q, df))

python
Most similar: Five Advanced Plots in Python - Matplotlib
Jaccard similarity: 0.167
plot neural network
Most similar: Training Provably-Robust Neural Networks
Jaccard similarity: 0.143
plot neural networks
Most similar: Training Provably-Robust Neural Networks
Jaccard similarity: 0.333
ploting neural networks
Most similar: Training Provably-Robust Neural Networks
Jaccard similarity: 0.333
data science
Most similar: CHAPTER 2: Why I Chose Data Science!
Jaccard similarity: 0.333


## Task 3

TFIDF (term frequency–inverse document frequency) is a much better approach. The tf–idf value increases proportionally to the number of times a word appears in the document and is offset by the number of documents in the corpus that contain the word, which helps to adjust for the fact that some words appear more frequently in general.

This approach consists of 2 steps:
TF (term frequency) -  $tf(t,d)$, is the relative frequency of term $t$ within document $d$, can be expressed e.g. as a word count divided by number of terms in a given document or by the maximum term count in a given document.

IDF (inverse document frequency) - is a measure of how much information the word provides. If a word appears in every document it does not provide much information, but if it just appears in two documents then its impact on similiarity between these two documents is higher. The standard approach to compute this value is logarithm of number of documents divided by number of documents containing a given term $IDF(t) = log(\frac{N}{n_t})$

TFIDF is then just TF multiplied by IDF


Implement tf idf, compare it with sklearn TfidfVectorizer

In [9]:
tfidf=TfidfVectorizer(use_idf=True, smooth_idf=False)

dfTFIDF = pd.DataFrame(tfidf.fit_transform(documents).toarray(), index=documents, columns=tfidf.get_feature_names_out())
dfTFIDF

,6x,about,achieve,advanced,ai,analysis,and,animated,anomaly,applications,...,vision,vs,way,what,why,will,with,workings,you,your
Machine Learning,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Five Advanced Plots in Python - Matplotlib,0.000000,0.000000,0.000000,0.450495,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
How to Make your Computer Talk with Python,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.249731,0.000000,0.000000,0.316067
Anomaly Detection on Servo Drives,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.459985,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Key takeaways from Kaggle’s most recent time series competition - Ventilator Pressure Prediction,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Animated Mathematical Analysis,0.000000,0.000000,0.000000,0.000000,0.000000,0.57735,0.000000,0.57735,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
How to Perform Speech Recognition with Python,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.280999,0.000000,0.000000,0.000000
Beyond The Semesters: E04,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"How to improve classification of e-commerce pages, incorporating multiple modalities",0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Time Series Forecasting with ThymeBoost,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.324860,0.000000,0.000000,0.000000


In [10]:
pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values()

of            2.491655
to            2.491655
with          2.609438
and           2.897120
how           2.897120
                ...   
ventilator    4.688879
variance      4.688879
what          4.688879
will          4.688879
workings      4.688879
Length: 184, dtype: float64

In [11]:
query = "how to machine learning"
query = tfidf.transform([query]).toarray()[0]
1-dfTFIDF.apply(lambda x: cosine(x, query), axis=1).sort_values()

Machine Learning                                                                                    0.763355
Recommender System With Machine Learning and Statistics                                             0.364334
Fastest Way of Deploying Your Machine Learning Models                                               0.328159
How to Perform Speech Recognition with Python                                                       0.265814
3 Lessons Learned in Teaching Machine Learning for Earth Observation Techniques                     0.258540
How to achieve efficient large-batch training?                                                      0.246288
How to Make your Computer Talk with Python                                                          0.236235
How to improve classification of e-commerce pages, incorporating multiple modalities                0.221282
How to improve classification of e-commerce pages, incorporating multiple modalities                0.221282
Exploring Methods o

In [12]:
# Step 1: Compute term frequency (TF)
tf = CountData.toarray() / CountData.toarray().sum(axis=1, keepdims=True)

# Step 2: Compute inverse document frequency (IDF)
N = len(documents)
df_count = np.count_nonzero(CountData.toarray() > 0, axis=0)
idf = np.log(N / df_count)

# Step 3: Combine to TF-IDF
manual_tfidf = tf * idf

# Convert to DataFrame for comparison
df_manual_tfidf = pd.DataFrame(manual_tfidf, columns=CountVec.get_feature_names_out(), index=documents)

# --- Compare with sklearn TF-IDF ---
tfidf = TfidfVectorizer(use_idf=True, smooth_idf=False)
df_tfidf_sklearn = pd.DataFrame(tfidf.fit_transform(documents).toarray(),
                                index=documents,
                                columns=tfidf.get_feature_names_out())

# Display IDF values (sklearn)
print("📊 Sklearn IDF values (sorted):")
print(pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values().head())

# --- Test query similarity using cosine distance ---
query = "how to machine learning"
query_vec = tfidf.transform([query]).toarray()[0]

similarities = 1 - df_tfidf_sklearn.apply(lambda x: cosine(x, query_vec), axis=1)
print("\n🔍 Cosine similarity results for query: \"how to machine learning\"")
print(similarities.sort_values(ascending=False).head())


📊 Sklearn IDF values (sorted):
of      2.491655
to      2.491655
with    2.609438
and     2.897120
how     2.897120
dtype: float64

🔍 Cosine similarity results for query: "how to machine learning"
Machine Learning                                                                   0.763355
Recommender System With Machine Learning and Statistics                            0.364334
Fastest Way of Deploying Your Machine Learning Models                              0.328159
How to Perform Speech Recognition with Python                                      0.265814
3 Lessons Learned in Teaching Machine Learning for Earth Observation Techniques    0.258540
dtype: float64


## Task 4
Create a search engine based on TFIDF

In [13]:
from scipy.spatial.distance import cosine

def search(query, dfTFIDF, tfidf, top_n=5):
    """
    Returns the top_n most similar documents to the query using TF-IDF and cosine similarity.
    
    Parameters:
    - query: str, the search query
    - dfTFIDF: pandas DataFrame, TF-IDF values for documents
    - tfidf: fitted TfidfVectorizer object
    - top_n: int, number of top results to return
    
    Returns:
    - pandas Series of top_n documents with similarity scores
    """
    # Transform query to TF-IDF vector
    query_vec = tfidf.transform([query]).toarray()[0]
    
    # Compute cosine similarity (1 - cosine distance)
    similarities = 1 - dfTFIDF.apply(lambda x: cosine(x, query_vec), axis=1)
    
    # Sort descending and return top_n
    return similarities.sort_values(ascending=False).head(top_n)


In [14]:
queries = [
    "python",
    "plot neural network",
    "data science",
    "speech recognition"
]

for q in queries:
    print(f"Query: {q}")
    print(search(q, dfTFIDF, tfidf))
    print("-" * 50)


Query: python
How to Perform Speech Recognition with Python                   0.355641
Five Advanced Plots in Python - Matplotlib                      0.317303
How to Make your Computer Talk with Python                      0.316067
1 Line of Python Code That Will Speed Up Your AI by Up to 6x    0.191295
Anomaly Detection on Servo Drives                               0.000000
dtype: float64
--------------------------------------------------
Query: plot neural network
Training Provably-Robust Neural Networks                                                            0.392352
Graph Neural Networks and Generalizable Models in Neuroscience                                      0.327037
How Parallelization and Large Batch Size Improve the Performance of Deep Neural Networks.           0.265386
Machine Learning                                                                                    0.000000
Key takeaways from Kaggle’s most recent time series competition - Ventilator Pressure Predic

## Task 5
Create a search engine based on history containing more than one document

In [15]:
from scipy.spatial.distance import cosine
import numpy as np

def search(history, dfTFIDF, tfidf, top_n=5):
    """
    Search for documents most similar to a list of query/history documents.
    
    Parameters:
    - history: list of strings (previous queries/documents)
    - dfTFIDF: pandas DataFrame with TF-IDF values for documents
    - tfidf: fitted TfidfVectorizer object
    - top_n: number of top results to return
    
    Returns:
    - pandas Series of top_n documents with similarity scores
    """
    # Transform each document in history to TF-IDF vector
    history_vecs = tfidf.transform(history).toarray()
    
    # Compute the mean vector of the history
    combined_vec = np.mean(history_vecs, axis=0)
    
    # Compute cosine similarity to all documents in dfTFIDF
    similarities = 1 - dfTFIDF.apply(lambda x: cosine(x, combined_vec), axis=1)
    
    # Sort descending and return top_n
    return similarities.sort_values(ascending=False).head(top_n)


In [16]:
history = [
    "python machine learning",
    "speech recognition with python"
]

print(search(history, dfTFIDF, tfidf))

How to Perform Speech Recognition with Python                                   0.606230
Machine Learning                                                                0.505124
Recommender System With Machine Learning and Statistics                         0.305007
How to Make your Computer Talk with Python                                      0.267354
A Novel Approach to Integrate Speech Recognition into Authentication Systems    0.217480
dtype: float64
